In [1]:
import os
import sys

os.chdir('../..')
# os.chdir('../MFdfn_v10')


In [2]:
sys.path.append("../../")
from emulator import preProcess
from emulator.utilsPyapprox import *
from emulator.utilsBayesianLinearRegression import *

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

np.random.seed(2)
# Imports
%matplotlib inline
%config InlineBackend.figure_format = 'svg'



In [3]:
def getGraphs(data_folder, quantile, fidelity, verbose):
    worker = preProcess(quantile=quantile, fidelity=fidelity, verbose=verbose)
    worker.build_graphs_from_dfn_data(data_folder)

    return worker


In [4]:
def getFeatures(worker, numShortestPath, flux_calc_pct, verbose):
    X_list, y = worker.extract_features_from_graphs(k_shortest_path=numShortestPath, flux_calc_pct=flux_calc_pct, verbose=verbose)

    return X_list, y



In [5]:
# build hi-fi graphs from data
# worker_hf = getGraphs('../../math-clinic-data/gpr_data_stream_1/', '50 percent', 'high', False)
# worker_hf = getGraphs('../../math-clinic-data/gpr_data_stream_2/', '50 percent', 'high', False)
# worker_hf = getGraphs('../../math-clinic-data/ds1/', '50 percent', 'high', False)
worker_hf = getGraphs('../../math-clinic-data/ds2/', '50 percent', 'high', False)
# worker_hf = getGraphs('../../math-clinic-data/ds3/', '50 percent', 'high', False)
# worker_hf = getGraphs('../../math-clinic-data/ds4/', '50 percent', 'high', False)
# worker_hf = getGraphs('../../math-clinic-data/ds5/', '50 percent', 'high', False)


Creating Graph Based on DFN
Intersections being mapped to nodes and fractures to edges
Graph Construction Complete
Solving sparse system
Updating graph nodes features with flow solution
Updating graph edges with flow solution
graph flow complete
Creating Graph Based on DFN
Intersections being mapped to nodes and fractures to edges
Graph Construction Complete
Solving sparse system
Updating graph nodes features with flow solution
Updating graph edges with flow solution
graph flow complete
Creating Graph Based on DFN
Intersections being mapped to nodes and fractures to edges
Graph Construction Complete
Solving sparse system
Updating graph nodes features with flow solution
Updating graph edges with flow solution
graph flow complete
Creating Graph Based on DFN
Intersections being mapped to nodes and fractures to edges
Graph Construction Complete
Solving sparse system
Updating graph nodes features with flow solution
Updating graph edges with flow solution
graph flow complete
Creating Graph B

/Users/jjkath/Library/CloudStorage/GoogleDrive-jjkath@gmail.com/My Drive/Programs/Python/math-clinic/MFdfn_v10/emulator/crawler/crawler.py:161: RuntimeWarning: divide by zero encountered in double_scalars
  sample_g.edges[u, v]['flux'] = (


Graph Construction Complete
Solving sparse system
Updating graph nodes features with flow solution
Updating graph edges with flow solution
graph flow complete
Creating Graph Based on DFN
Intersections being mapped to nodes and fractures to edges
Graph Construction Complete
Solving sparse system
Updating graph nodes features with flow solution
Updating graph edges with flow solution
graph flow complete
Creating Graph Based on DFN
Intersections being mapped to nodes and fractures to edges
Graph Construction Complete
Solving sparse system
Updating graph nodes features with flow solution
Updating graph edges with flow solution
graph flow complete
Creating Graph Based on DFN
Intersections being mapped to nodes and fractures to edges
Graph Construction Complete
Solving sparse system
Updating graph nodes features with flow solution
Updating graph edges with flow solution
graph flow complete
Creating Graph Based on DFN
Intersections being mapped to nodes and fractures to edges
Graph Constructi

In [6]:
# build lo-fi graphs from data
# worker_lf = getGraphs('../../math-clinic-data/gpr_data_stream_1/', '50 percent', 'low', False)
# worker_lf = getGraphs('../../math-clinic-data/gpr_data_stream_2/', '50 percent', 'low', False)
# worker_lf = getGraphs('../../math-clinic-data/ds1/', '50 percent', 'low', False)
worker_lf = getGraphs('../../math-clinic-data/ds2/', '50 percent', 'low', False)
# worker_lf = getGraphs('../../math-clinic-data/ds3/', '50 percent', 'low', False)
# worker_lf = getGraphs('../../math-clinic-data/ds4/', '50 percent', 'low', False)
# worker_lf = getGraphs('../../math-clinic-data/ds5/', '50 percent', 'low', False)


Creating Graph Based on DFN
Intersections being mapped to nodes and fractures to edges
Graph Construction Complete
Solving sparse system
Updating graph nodes features with flow solution
Updating graph edges with flow solution
graph flow complete
Creating Graph Based on DFN
Intersections being mapped to nodes and fractures to edges
Graph Construction Complete
Solving sparse system
Updating graph nodes features with flow solution
Updating graph edges with flow solution
graph flow complete
Creating Graph Based on DFN
Intersections being mapped to nodes and fractures to edges
Graph Construction Complete
Solving sparse system
Updating graph nodes features with flow solution
Updating graph edges with flow solution
graph flow complete
Creating Graph Based on DFN
Intersections being mapped to nodes and fractures to edges
Graph Construction Complete
Solving sparse system
Updating graph nodes features with flow solution
Updating graph edges with flow solution
graph flow complete
Creating Graph B

In [7]:
# get hi-fi features from graphs
k = 100  # 1 3 (5*) 10 15 (25) 50 ... 100 1000
flux_calc_pct = 50      # (55) (59 59)[k=5] (44..57 57)[k=100] 55 .. 25 .. 50
X_hf_list, y_hf = getFeatures(worker_hf, k, flux_calc_pct, False)
# log y_dfn
y_log_hf = np.log(np.array(y_hf.reshape(-1, 1), dtype='float64'))


In [8]:
# get lo-fi features from graphs
k = 100 # 1000
flux_calc_pct = 50      # (10) (16 13)[k=5] 10 .. 25 .. 50
X_lf_list, y_lf = getFeatures(worker_lf, k, flux_calc_pct, False)
# log y_dfn
y_log_lf = np.log(np.array(y_lf.reshape(-1, 1), dtype='float64'))


In [9]:
# LSF line to y_graph (x-axis) vs y_dfn (y-axis)
reg = LinearRegression().fit(y_log_lf[0:len(y_log_hf)], y_log_hf)
# reg = LinearRegression().fit(y_log_lf, y_log_hf)
reg_m = reg.coef_[0][0]
reg_b = reg.intercept_[0]

print(f'score(bias): {reg.score(y_log_lf[0:len(y_log_hf)], y_log_hf):.4f}')
print(f'm(bias): {reg_m:.4f}')
print(f'b(bias): {reg_b:.4f}')

# remove bias
y_log_lf_ub = reg_m * y_log_lf.copy() + reg_b
# y_log_lf_ub = (1/reg_m) * (y_log_lf - reg_b)

# y_log_hf_trans = reg_m * y_log_hf.copy() + reg_b

# check correction for bias
reg_ub = LinearRegression().fit(y_log_lf_ub[0:len(y_log_hf)], y_log_hf)
reg_m_ub = reg_ub.coef_[0][0]
reg_b_ub = reg_ub.intercept_[0]

print(f'score(unbiased): {reg_ub.score(y_log_lf_ub[0:len(y_log_hf)], y_log_hf):.4f}')
print(f'm(unbiased): {reg_m_ub:.4f}')
print(f'b(unbiased): {reg_b_ub:.4f}')

# reg.predict(np.array([[3, 5]]))


score(bias): 0.7662
m(bias): 0.5975
b(bias): 3.8702
score(unbiased): 0.7662
m(unbiased): 1.0000
b(unbiased): 0.0000


In [10]:
# calculate absolute raw and percent errors
y_pred_error = np.abs(np.exp(y_log_lf_ub[0:len(y_log_hf)]) - np.exp(y_log_hf))
# y_pred_error = np.abs(np.exp(y_log_lf_ub) - np.exp(y_log_hf))
y_pred_percent_error = 100 * (y_pred_error / np.exp(y_log_hf))

pd.DataFrame(y_pred_percent_error, columns=['median breakthrough time hi-fi (all y_dfn vs y_graph(unbiased)), '
                                           f'abs percent difference: |y_graph_ub - y_dfn| / y_dfn']).describe()


,"median breakthrough time hi-fi (all y_dfn vs y_graph(unbiased)), abs percent difference: |y_graph_ub - y_dfn| / y_dfn"
count,100.000000
mean,21.007767
std,16.163708
min,0.081495
25%,7.120227
50%,17.227788
75%,30.212500
max,67.609980


In [11]:
# save copy of X
# X_hfm = X_hf.copy()
X_hfm_list, X_lfm_list = X_hf_list.copy(), X_lf_list.copy()


In [12]:
# use network features as needed
# feature_list = [path_len, flux_pct, iperm_sum, time_pct, length_sum]
# feature_list = [path_len 30,    flux_pct 31/30, iperm_sum 33/29, time_pct 42/41,  length_sum 48] / ds1 hi-fi k=1000 (alpha)
# feature_list = [path_len 26/27, flux_pct 28,    iperm_sum 28/27, time_pct 425/41, length_sum 43] / ds2 hi-fi k=1000 (beta)
# feature_list = [path_len 29,    flux_pct 31/30, iperm_sum 32/28, time_pct 44/41,  length_sum 45] / ds1 hi-fi k=100  (delta)
# feature_list = [path_len 26/27, flux_pct 27/26, iperm_sum 28,    time_pct 303/40, length_sum 42] / ds2 hi-fi k=100  (gamma)
# single feature
feature_index = 0
# X_hf = X_hfm[:,feature_index].reshape(-1,1)
X_hf, X_lf = X_hfm[:,feature_index].reshape(-1,1), X_lfm[:,feature_index].reshape(-1,1)
# X_hf, X_lf = np.hstack([X_hfm[:,feature_index].reshape(-1,1), X_hfm[:,feature_index+5].reshape(-1,1)]), np.hstack([X_lfm[:,feature_index].reshape(-1,1), X_lfm[:,feature_index+5].reshape(-1,1)])
# multi feature
# X_hf = np.hstack([X_hfm[:,0].reshape(-1,1), X_hfm[:,1].reshape(-1,1)])
# X_hf = np.hstack([X_hfm[:,0].reshape(-1,1), X_hfm[:,1].reshape(-1,1), X_hfm[:,2].reshape(-1,1)])
# # X_lf = np.hstack([X_lfm[:,1].reshape(-1,1), X_lfm[:,3].reshape(-1,1)])
# X_lf = np.hstack([X_lfm[:,0].reshape(-1,1), X_lfm[:,1].reshape(-1,1), X_lfm[:,2].reshape(-1,1)])
# all features
# X_hf = X_hfm
# X_lf = X_lfm



NameError: name 'X_hfm' is not defined

In [12]:
# gn cross fold validation (hi-fi)
nfolds = 10 # 10
swap = False # True False

# poly degree
degree_cfv = 2      # 1 (2) 3
# poly sigma
sigma_cfv = 1.0     # 1.0 (3.0)
# glm precision
beta_cfv  = 15.5     # 15.5 20.5        [50th pct]
alpha_cfv = 1       # 8 (k=3) 8.5 (k=100) | 7.5 (k=3) 7 (k=100)     [90th pct]
# beta_cfv  = beta
# alpha_cfv = alpha

# training data samples (X) + values (y)
k = 100
X_cv = X_hf_list[k-1].copy()
y_cv = y_log_hf.copy()
# y_cv = y_log_hf_trans.copy()

y_pred, y_pred_var, y_test = gn_cv(X_cv, y_cv, degree=degree_cfv, sigma=sigma_cfv, alpha=alpha_cfv, beta=beta_cfv, nfolds=nfolds, swap_folds=swap)

# # y_inv_trans = (1/reg_m) * (y_trans - reg_b)
# y_pred = (1/reg_m) * (y_pred - reg_b)
# y_test = (1/reg_m) * (y_test - reg_b)

# calculate absolute raw and percent errors
y_pred_error = np.abs(np.exp(y_pred) - np.exp(y_test))
y_pred_percent_error = 100 * (y_pred_error / np.exp(y_test))

# print(f'pct: {ii}, hi-fi MAPE: {np.mean(y_pred_percent_error):.2f}')

swap_s = '_sw' if swap else ''
# pd.set_option("max_colwidth", 10)
pd.DataFrame(y_pred_percent_error, columns=['median breakthrough time hi-fi (testing only), '
                                           f'abs percent difference: |y_sfnet_{nfolds}fcv{swap_s} - y_dfn| / y_dfn']).describe()
# df.describe()


,"median breakthrough time hi-fi (testing only), abs percent difference: |y_sfnet_10fcv - y_dfn| / y_dfn"
count,100.000000
mean,27.446001
std,25.703275
min,0.307639
25%,8.938594
50%,22.129614
75%,37.682318
max,155.434123


In [13]:
# percent observations withing 95% confidence intervals
y = np.exp(y_test)
y_pred_std = np.sqrt(y_pred_var)
y_lower = np.exp(y_pred-1.96*y_pred_std)
y_upper = np.exp(y_pred+1.96*y_pred_std)
isInCI = np.logical_and(y >= y_lower,y <= y_upper)
print(f'number of observations in 95% CI: {np.count_nonzero(isInCI)} / {len(y_pred)}, {(100*np.count_nonzero(isInCI)/len(y_pred)):.2f}')
# y_pred_std
pd.DataFrame(np.exp(y_pred_std), columns=['median breakthrough time hi-fi (testing only), '
                                          'y_predicted standard deviation']).describe()


number of observations in 95% CI: 88 / 100, 88.00


,"median breakthrough time hi-fi (testing only), y_predicted standard deviation"
count,100.000000
mean,1.289475
std,0.000337
min,1.289310
25%,1.289328
50%,1.289357
75%,1.289468
max,1.291538


In [14]:
# gn cross fold validation (lo-fi)
nfolds = 10
swap = False # True False

# poly degree
degree_cfv = 2      # 1 2 (3)
# poly sigma
sigma_cfv = 1.0     # 1.0 (3.0)
# glm precision
beta_cfv  = 15.5    # (15.5) (5.5)           [50th pct]
alpha_cfv = 1       # 3.05 (k=3) 3.4 (k=100) | 3.73 (k=3) 3.93 (k=100)      [90th pct]
# beta_cfv  = beta
# alpha_cfv = alpha

# training data samples (X) + values (y)
k = 100
X_cv = X_lf_list[k-1].copy()
# y_cv = y_log_lf.copy()
y_cv = y_log_lf_ub.copy()

y_pred, y_pred_var, y_test = gn_cv(X_cv, y_cv, degree=degree_cfv, sigma=sigma_cfv, alpha=alpha_cfv, beta=beta_cfv, nfolds=nfolds, swap_folds=swap)

# calculate absolute raw and percent errors
y_pred_error = np.abs(np.exp(y_pred) - np.exp(y_test))
y_pred_percent_error = 100 * (y_pred_error / np.exp(y_test))

# print(f'pct: {ii}, lo-fi MAPE: {np.mean(y_pred_percent_error):.2f}')

swap_s = '_sw' if swap else ''
pd.DataFrame(y_pred_percent_error, columns=['median breakthrough time lo-fi (testing only), '
                                           f'abs percent difference: |y_sfnet_{nfolds}fcv{swap_s} - y_dfn| / y_dfn']).describe()


,"median breakthrough time lo-fi (testing only), abs percent difference: |y_sfnet_10fcv - y_dfn| / y_dfn"
count,100.000000
mean,26.398335
std,19.995101
min,0.654835
25%,12.238522
50%,20.388153
75%,37.238036
max,98.878808


In [15]:
# percent observations withing 95% confidence intervals
y = np.exp(y_test)
y_pred_std = np.sqrt(y_pred_var)
y_lower = np.exp(y_pred-1.96*y_pred_std)
y_upper = np.exp(y_pred+1.96*y_pred_std)
isInCI = np.logical_and(y >= y_lower,y <= y_upper)
print(f'number of observations in 95% CI: {np.count_nonzero(isInCI)} / {len(y_pred)}, {(100*np.count_nonzero(isInCI)/len(y_pred)):.2f}')
# y_pred_std
pd.DataFrame(np.exp(y_pred_std), columns=['median breakthrough time lo-fi (testing only), '
                                          'y_predicted standard deviation']).describe()



number of observations in 95% CI: 90 / 100, 90.00


,"median breakthrough time lo-fi (testing only), y_predicted standard deviation"
count,100.000000
mean,1.289475
std,0.000337
min,1.289310
25%,1.289328
50%,1.289357
75%,1.289468
max,1.291538


In [ ]:
# poly degree
degree_h = 2 # 1 2 3
degree_l = 2 # 1 2 3
# glm precision
beta_h  = 5.5     # 10.0 1.26 8 9
beta_l  = 25.5    #  6.5
alpha_h = 1       #  0.05
alpha_l = 1       #  0.005


In [ ]:
# hi-fi training data samples (X) + values (y)
k = 100
X_h = X_hf_list[k-1].copy()
X_h = X_h / X_h.max(axis=0) # norm perm / iperm
y_h = y_log_hf.copy()


In [ ]:
# lo-fi training data samples (X) + values (y)
k = 100
X_l = X_lf_list[k-1].copy()
X_l = X_l / X_l.max(axis=0) # norm perm / iperm
y_l = y_log_lf.copy()


In [ ]:
# set random seed for training data
rng = np.random.RandomState(1)


In [ ]:
# choose training & testing data
X, y = [X_l.copy(), X_h.copy()], [y_l.copy(), y_h.copy()]

# choose all data for training
# X_train, y_train = X, y

# choose random samples / values for training
X_train, y_train, index_train = choose_sample_value_rnd(X, y, lf_pct=1.0, hf_pct=0.15, rng=rng)

# choose quantile samples / values for training
# X_train, y_train, index_train = choose_sample_value_pct(X, y, rng=rng)


In [ ]:
# construct the feature matrix
Phi_train_h = pce_basis_matrix(X_train[1], degree=degree_h, dist_bounds=[-15, 15])
Phi_train_l = pce_basis_matrix(X_train[0], degree=degree_l, dist_bounds=[-15, 15])
Phi_test_h = pce_basis_matrix(X[1], degree=degree_h, dist_bounds=[-15, 15])
Phi_test_l = pce_basis_matrix(X[0], degree=degree_l, dist_bounds=[-15, 15])


In [ ]:
# Design matrix of test observations
Phi_train_h = gn_expand(X_train[1], bf=gn_polynomial_basis_function, bf_deg_args=np.linspace(0, 1, degree_h+1)[1:])
Phi_train_l = gn_expand(X_train[0], bf=gn_polynomial_basis_function, bf_deg_args=np.linspace(0, 1, degree_l+1)[1:])
Phi_test_h = gn_expand(X[1], bf=gn_polynomial_basis_function, bf_deg_args=np.linspace(0, 1, degree_h+1)[1:])
Phi_test_l = gn_expand(X[0], bf=gn_polynomial_basis_function, bf_deg_args=np.linspace(0, 1, degree_l+1)[1:])
# Phi_train_h = gn_expand(X_train[1], bf=gn_polynomial_basis_function, bf_deg_args=range(1, degree_h + 1))
# Phi_train_l = gn_expand(X_train[0], bf=gn_polynomial_basis_function, bf_deg_args=range(1, degree_l + 1))
# Phi_test_h = gn_expand(X[1], bf=gn_polynomial_basis_function, bf_deg_args=range(1, degree_h + 1))
# Phi_test_l = gn_expand(X[0], bf=gn_polynomial_basis_function, bf_deg_args=range(1, degree_l + 1))


In [ ]:
# Design matrix of test observations
Phi_train_h = gn_expand(X_train[1], bf=gn_gaussian_basis_function, bf_deg_args=np.linspace(0, 1, degree_h), sigma=[2.0])
Phi_train_l = gn_expand(X_train[0], bf=gn_gaussian_basis_function, bf_deg_args=np.linspace(0, 1, degree_l), sigma=[2.0])
Phi_test_h = gn_expand(X[1], bf=gn_gaussian_basis_function, bf_deg_args=np.linspace(0, 1, degree_h), sigma=[2.0])
Phi_test_l = gn_expand(X[0], bf=gn_gaussian_basis_function, bf_deg_args=np.linspace(0, 1, degree_l), sigma=[2.0])


In [ ]:
# Mean and covariance matrix of posterior
post_mean_h, post_cov_h = gn_posterior(Phi_train_h, y_train[1], alpha=alpha_h, beta=beta_h)
post_mean_l, post_cov_l = gn_posterior(Phi_train_l, y_train[0], alpha=alpha_l, beta=beta_l)


In [ ]:
# Mean and variances of posterior predictive 
y_pred_h, y_pred_var_h = gn_posterior_predictive(Phi_test_h, post_mean_h, post_cov_h, beta=beta_h)
y_pred_l, y_pred_var_l = gn_posterior_predictive(Phi_test_l, post_mean_l, post_cov_l, beta=beta_l)


In [ ]:
# MAPE on training only (samples, data)
values = y[1].reshape(-1,1)
post_mean_y_pred = y_pred_h
# values = y[0].reshape(-1,1)
# post_mean_y_pred = y_pred_l

# calculate absolute raw and percent errors
post_mean_errors = np.abs(np.exp(post_mean_y_pred) - np.exp(values))
post_mean_percent_errors_train = 100 * (post_mean_errors / np.exp(values))

pd.DataFrame(post_mean_percent_errors_train, columns=['median breakthrough time hi-fi (testing and training), '
                                                      'abs percent difference: |y_sfnet - y_dfn| / y_dfn']).describe()
# pd.DataFrame(post_mean_percent_errors_train, columns=['median breakthrough time lo-fi (testing and training), '
#                                                       'abs percent difference: |y_sfnet - y_dfn| / y_dfn']).describe()
# #%%
# # verify valid CI
# # percent observations withing 95% confidence intervals
# values = np.exp(y_test)
# y_pred_std = np.sqrt(y_pred_var)
# y_lower = np.exp(y_pred-1.96*y_pred_std)
# y_upper = np.exp(y_pred+1.96*y_pred_std)
# isInCI = np.logical_and(values >= y_lower,values <= y_upper)
# print(f'number of observations in 95% CI: {np.count_nonzero(isInCI)} / {len(y_pred)}, {(100*np.count_nonzero(isInCI)/len(y_pred)):.2f}')
# # y_pred_std
# pd.DataFrame(np.exp(y_pred_std), columns=['median breakthrough time hi-fi (testing only), '
#                                           'y_predicted standard deviation']).describe()

In [ ]:
# y_pred_var
post_mean_y_var = y_pred_var_h
pd.DataFrame(np.exp(post_mean_y_var), columns=['median breakthrough time hi-fi (testing and training), '
                                               'log y_predicted variance']).describe()



In [ ]:
# plot sfnet prediction
degree, post_mean, post_cov, beta = degree_h, post_mean_h, post_cov_h, beta_h
samples_train, values_train = [X_train[1].T.copy()], [y_train[1].copy()]
samples_test, values_test = [X[1].T.copy()], [y[1]]
y_ranges=[7, 13.5] # [8, 12.5] [6, 11.5] [np.exp(6), np.exp(11.5)]
x_ranges=[0, 1]
# degree, post_mean, post_cov, beta = degree_l, post_mean_l, post_cov_l, beta_l
# samples_train, values_train = [X_train[0].T.copy()], [y_train[0].copy()]
# samples_test, values_test = [X[0].T.copy()], [y[0]]
# y_ranges=[6, 14.5] # 
# x_ranges=[0, 1]
y_log = True # True False

dim1_index = 0

xx = np.linspace(0, 1, 100)
fig, axs = plt.subplots(1, 1, figsize=(8, 8)) # (6, 6)
training_labels = [r'$\mathrm{{log\: y\: dfn}_{hi-fi}\: train}$', r'$\mathrm{{log\: y\: dfn}_{hi-fi}\: test}$']
# training_labels = [r'$\mathrm{log\: y\: dfn}_{lo-fi}$']
# training_labels = [r'$\mathrm{y\: dfn}_{hi-fi}$']
# training_labels = [r'$\mathrm{y\: dfn}_{lo-fi}$']
plot_nd_sf_lvn_approx(xx, axs, degree, post_mean, post_cov, beta,
                   samples_train, values_train, samples_test, values_test, training_labels,
                   dim1_index, x_ranges=x_ranges, y_ranges=y_ranges, y_log=y_log,
                   pct=[50],
                  #  pct=[25, 75, 50],
                  #  pct=[10, 90, 25, 75, 50], # [25, 50, 75] [10, 25, 50, 75, 90]
                   colors=['tab:orange'], # b c r k tab:orange
                   colors_pct=['green'])
                  #  colors_pct=['turquoise', 'paleturquoise', 'green'])
                  #  colors_pct=['lavender', 'lightcyan', 'turquoise', 'paleturquoise', 'green'])
# axs.set_xlabel(r'$\mathrm{network\: iperm\: /\: max\: iperm}$')
# axs.set_xlabel(r'$\mathrm{network\: fracture\: length\: /\: max\: length}$')
# axs.set_xlabel(r'$\mathrm{network\: mass\: flux\: /\: max\: flux}$')
# axs.set_xlabel(r'$\mathrm{network\: travel\: time\: /\: max\: time}$')
axs.set_xlabel(r'$\mathrm{network\: path\: length\: /\: max\: length}$')
axs.set_ylabel(r'$\mathrm{median\: breakthrough\: time}$')
plt.show()



In [ ]:
# evaluate the log marginal likelihood for all polynomials up to specified degree
# goal is to find poly degree which minimizes model error
mlls = []
degree_test = 5 # 2 .. 5
degrees = range(degree_test + 1)
X_dt, y_dt = X_h, y_h
# beta_test  = 11.11 # 5.0 1 / (0.3 ** 2) = 11.11 beta
# alpha_test = 0.005 # 2.0 0.005 alpha
beta_test  = beta
alpha_test = alpha

for d in degrees:
   #  Phi_test = pce_basis_matrix(X_dt, degree=d, dist_bounds=[0, 1])
    Phi_test = gn_expand(X_dt, bf=gn_gaussian_basis_function, bf_deg_args=np.linspace(0, 1, d))
    mll = gn_log_marginal_likelihood(Phi_test, y_dt, alpha=alpha_test, beta=beta_test)
   #  mll = blr_log_marginal_likelihood(Phi_test, y_dt, alpha=alpha_test, beta=alpha_test)
    mlls.append(mll)

degree_max = np.argmax(mlls)
    
plt.plot(degrees, mlls)
plt.axvline(x=degree_max, ls='--', c='k', lw=1)
plt.xticks(range(0, degree_test+1))
plt.xlabel('Polynomial degree')
plt.ylabel('Log marginal likelihood');



In [ ]:
# evaluate the log marginal likelihood for all polynomials of given degree 
# goal is to find poly sigma which minimizes model error
mlls = []
degree_test = 2
sigma_test = 5
sigmas = np.linspace(0.1, sigma_test, sigma_test*10)
# sigmas = range(1, sigma_test + 1)
X_dt, y_dt = X_h, y_h
beta_test  = 15
alpha_test = 1
# beta_test  = beta
# alpha_test = alpha

for s in sigmas:
    Phi_test = gn_expand(X_dt, bf=gn_gaussian_basis_function, bf_deg_args=np.linspace(0, 1, d), sigma=[s])
    mll = gn_log_marginal_likelihood(Phi_test, y_dt, alpha=alpha_test, beta=beta_test)
    mlls.append(mll)

sigma_max = np.argmax(mlls)
print(sigmas[sigma_max])
    
plt.plot(sigmas, mlls)
plt.axvline(x=sigmas[sigma_max], ls='--', c='k', lw=1)
plt.xticks(np.linspace(0.1, sigma_test, 5))
plt.xlabel('Sigma')
plt.ylabel('Log marginal likelihood');



In [ ]:
# use fit to obtain the posterior over parameters 𝐰 and optimal values for 𝛼 and 𝛽
degree_test = 2 # 1 2 3
X_dt, y_dt = X_h, y_h
# Phi_test = pce_basis_matrix(X_dt, degree=degree_test, dist_bounds=[0, 1])
Phi_test = expand(X_dt, bf=gaussian_basis_function, bf_args=np.linspace(0, 1, degree_test))
alpha, beta, m_N, S_N = gn_fit(Phi_test, y_dt, rtol=1e-5, verbose=True)
# alpha, beta, m_N, S_N = blr_fit(Phi_test, y_dt, rtol=1e-5, verbose=True)

print(f'alpha: {alpha}')
print(f'beta: {beta}')

